<a href="https://colab.research.google.com/github/2303A51589/RL/blob/main/assi_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Value Iteration in a Deterministic 2×2 Grid World
This notebook implements the Value Iteration algorithm to find the optimal state values and optimal policy for a 2x2 grid world.

In [1]:
import pandas as pd
import numpy as np

# --- 1. Define the MDP environment ---
states = ['S0', 'S1', 'G']
non_terminal_states = ['S0', 'S1']
actions = ['Up', 'Down', 'Left', 'Right']
gamma = 0.9

# Transition and reward rules defined by the 2x2 grid layout:
# S0 | Wall
# S1 | G (Terminal)
# If movement hits wall or boundary, agent stays in current state with reward -1.
# Reaching G gives reward +100.

def transition_and_reward(state, action):
    if state == 'G':
        return 'G', 0  # Terminal state

    if state == 'S0':
        if action == 'Down':
            next_state = 'S1'
            reward = -1
        else:
            # Up, Left, Right hit boundaries/wall
            next_state = 'S0'
            reward = -1

    elif state == 'S1':
        if action == 'Right':
            next_state = 'G'
            reward = 100
        elif action == 'Up':
            next_state = 'S0'
            reward = -1
        else:
            # Down, Left hit boundaries
            next_state = 'S1'
            reward = -1

    return next_state, reward

# Print MDP configuration
print("=== MDP Configuration ===")
for s in non_terminal_states:
    for a in actions:
        ns, r = transition_and_reward(s, a)
        print(f"State: {s} | Action: {a:5} | Next State: {ns} | Reward: {r}")

=== MDP Configuration ===
State: S0 | Action: Up    | Next State: S0 | Reward: -1
State: S0 | Action: Down  | Next State: S1 | Reward: -1
State: S0 | Action: Left  | Next State: S0 | Reward: -1
State: S0 | Action: Right | Next State: S0 | Reward: -1
State: S1 | Action: Up    | Next State: S0 | Reward: -1
State: S1 | Action: Down  | Next State: S1 | Reward: -1
State: S1 | Action: Left  | Next State: S1 | Reward: -1
State: S1 | Action: Right | Next State: G | Reward: 100


In [5]:
# --- 2. Initialize Parameters and Run Value Iteration ---
V = {s: 0.0 for s in states}  # V(G) is terminal and remains 0.0
theta = 0.0001
iteration = 0
history = []

print("\n=== Iteration-wise State-Value Table ===")
print(f"Iteration {iteration}: V(S0) = {V['S0']:.4f}, V(S1) = {V['S1']:.4f}, V(G) = {V['G']:.4f}")
history.append({'Iteration': iteration, 'V(S0)': V['S0'], 'V(S1)': V['S1']})

while True:
    iteration += 1
    delta = 0
    V_new = V.copy()

    for s in non_terminal_states:
        v_old = V[s]
        q_values = []
        for a in actions:
            next_state, reward = transition_and_reward(s, a)
            q_value = reward + gamma * V[next_state]
            q_values.append(q_value)

        V_new[s] = max(q_values)
        delta = max(delta, abs(v_old - V_new[s]))

    V = V_new
    print(f"Iteration {iteration}: V(S0) = {V['S0']:.4f}, V(S1) = {V['S1']:.4f}, V(G) = {V['G']:.4f} (Delta: {delta:.4f})")
    history.append({'Iteration': iteration, 'V(S0)': V['S0'], 'V(S1)': V['S1']})

    if delta < theta:
        break

# Display history as table
df_history = pd.DataFrame(history)
display(df_history)


=== Iteration-wise State-Value Table ===
Iteration 0: V(S0) = 0.0000, V(S1) = 0.0000, V(G) = 0.0000
Iteration 1: V(S0) = -1.0000, V(S1) = 100.0000, V(G) = 0.0000 (Delta: 100.0000)
Iteration 2: V(S0) = 89.0000, V(S1) = 100.0000, V(G) = 0.0000 (Delta: 90.0000)
Iteration 3: V(S0) = 89.0000, V(S1) = 100.0000, V(G) = 0.0000 (Delta: 0.0000)


,Iteration,V(S0),V(S1)
0,0,0.0,0.0
1,1,-1.0,100.0
2,2,89.0,100.0
3,3,89.0,100.0


In [6]:
# --- 3. Compute Final Action-Values Q*(s, a) and Optimal Policy ---
q_table = []
optimal_policy = {}

for s in non_terminal_states:
    best_action = None
    best_q = -float('inf')
    for a in actions:
        next_state, reward = transition_and_reward(s, a)
        q_value = reward + gamma * V[next_state]
        q_table.append({'State': s, 'Action': a, 'Next State': next_state, 'Reward': reward, 'Q*(s,a)': round(q_value, 2)})

        if q_value > best_q:
            best_q = q_value
            best_action = a

    optimal_policy[s] = (best_action, round(best_q, 2))

print("\n=== Final Action-Value Table Q*(s, a) ===")
df_q = pd.DataFrame(q_table)
display(df_q)

print("\n=== Final Optimal State Values V*(s) & Policy ===")
for s in non_terminal_states:
    action, val = optimal_policy[s]
    print(f"V*({s}) = {val} | Optimal Action: {action}")
print(f"V*(G)  = 0 (Terminal)")


=== Final Action-Value Table Q*(s, a) ===


,State,Action,Next State,Reward,"Q*(s,a)"
0,S0,Up,S0,-1,79.1
1,S0,Down,S1,-1,89.0
2,S0,Left,S0,-1,79.1
3,S0,Right,S0,-1,79.1
4,S1,Up,S0,-1,79.1
5,S1,Down,S1,-1,89.0
6,S1,Left,S1,-1,89.0
7,S1,Right,G,100,100.0



=== Final Optimal State Values V*(s) & Policy ===
V*(S0) = 89.0 | Optimal Action: Down
V*(S1) = 100.0 | Optimal Action: Right
V*(G)  = 0 (Terminal)
